In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 300

distance = rng.uniform(10, 1000, n)
weight = rng.uniform(1, 500, n)

traffic = rng.choice(
    ["Low", "Medium", "High"],
    size=n,
    p=[0.35, 0.40, 0.25]
)

mode = rng.choice(
    ["Road", "Rail", "Air"],
    size=n,
    p=[0.60, 0.25, 0.15]
)

weather_delay = rng.choice(
    [0, 1, 2],
    size=n,
    p=[0.65, 0.25, 0.10]
)

speed = pd.Series(mode).map({
    "Road": 45,
    "Rail": 70,
    "Air": 400
}).to_numpy()

traffic_effect = pd.Series(traffic).map({
    "Low": 0.5,
    "Medium": 1.5,
    "High": 3.0
}).to_numpy()

mode_handling = pd.Series(mode).map({
    "Road": 1.0,
    "Rail": 2.0,
    "Air": 2.5
}).to_numpy()

noise = rng.normal(0, 1.2, n)

delivery_time = (
    distance / speed
    + traffic_effect
    + weather_delay * 1.5
    + mode_handling
    + noise
)

delivery_time = np.maximum(delivery_time, 0.5)

df = pd.DataFrame({
    "Distance_km": distance,
    "Shipment_Weight_kg": weight,
    "Traffic_Level": traffic,
    "Transport_Mode": mode,
    "Weather_Delay": weather_delay,
    "Delivery_Time_hr": delivery_time
})

df.to_csv(
    "Week_4_Logistics_Simulated_Data.csv",
    index=False
)

print("Dataset created successfully!")
print(df.head())
print("\nDataset shape:", df.shape)

Dataset created successfully!
   Distance_km  Shipment_Weight_kg Traffic_Level Transport_Mode  \
0   776.216488          389.719181        Medium           Road   
1   444.489655           68.141552          High           Road   
2   860.011941          268.497950          High           Road   
3   700.394349          257.597212        Medium           Road   
4   103.235574          428.928500          High            Air   

   Weather_Delay  Delivery_Time_hr  
0              0         20.904808  
1              0         13.008015  
2              1         23.586865  
3              1         20.115966  
4              0          6.944353  

Dataset shape: (300, 6)


In [4]:
X = df.drop(columns=["Delivery_Time_hr"])
y = df["Delivery_Time_hr"]

numeric_features = [
    "Distance_km",
    "Shipment_Weight_kg",
    "Weather_Delay"
]

categorical_features = [
    "Traffic_Level",
    "Transport_Mode"
]

print("Features and target prepared successfully!")

Features and target prepared successfully!


In [5]:
numeric_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created!")

Preprocessing pipeline created!


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 240
Testing records: 60


In [7]:
models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        min_samples_leaf=2
    )
}

results = []
trained_models = {}

for name, model in models.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    trained_models[name] = pipeline

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results)

print(results_df.round(3))

               Model    MAE   RMSE     R2
0  Linear Regression  1.855  2.437  0.866
1      Random Forest  1.432  1.770  0.929


In [8]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for name, model in models.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error"
    )

    fold_mae = -scores

    cv_results.append({
        "Model": name,
        "Mean_CV_MAE": fold_mae.mean(),
        "Std_CV_MAE": fold_mae.std()
    })

cv_df = pd.DataFrame(cv_results)

print(cv_df.round(3))

               Model  Mean_CV_MAE  Std_CV_MAE
0  Linear Regression        1.977       0.448
1      Random Forest        1.353       0.192


In [9]:
new_shipment = pd.DataFrame({
    "Distance_km": [250],
    "Shipment_Weight_kg": [80],
    "Traffic_Level": ["High"],
    "Transport_Mode": ["Road"],
    "Weather_Delay": [1]
})

# Choose the model with the lower test MAE.
best_model_name = results_df.loc[
    results_df["MAE"].idxmin(),
    "Model"
]

best_model = trained_models[best_model_name]

predicted_time = best_model.predict(
    new_shipment
)[0]

print("Selected model:", best_model_name)

print(
    "Predicted delivery time:",
    round(predicted_time, 2),
    "hours"
)

Selected model: Random Forest
Predicted delivery time: 10.4 hours


In [10]:
priority_threshold = 12

new_shipment["Predicted_Delivery_Time_hr"] = (
    best_model.predict(new_shipment)
)

new_shipment["Priority"] = np.where(
    new_shipment["Predicted_Delivery_Time_hr"]
    > priority_threshold,
    "Review",
    "Normal"
)

print(new_shipment)

   Distance_km  Shipment_Weight_kg Traffic_Level Transport_Mode  \
0          250                  80          High           Road   

   Weather_Delay  Predicted_Delivery_Time_hr Priority  
0              1                   10.397501   Normal  


In [11]:
from google.colab import files

files.download("Week_4_Logistics_Simulated_Data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>